# 12 · Frequency-offset estimation

The transmitter laser and the LO differ in frequency by up to a few GHz,
and the received constellation spins
([10](10-front-end-impairments.ipynb) · section 4).  Frequency-offset estimation (FOE) measures
that offset so that `correct_frequency_offset` can remove it before carrier
recovery handles the slow remainder.  The estimator is chosen by an object:

| method | principle | needs | lock range |
|---|---|---|---|
| `MthPower()` | `x^M` removes the modulation; FFT peak at `M·Δf` | nothing (blind) | `±fs / (2M)` |
| `MengaliMorelli()` | phase of the autocorrelation of `x^M` over many lags | nothing (blind) | `±fs / (2M)` |
| `PilotSymbols(indices, values)` | slope of the phase of known symbols | pilots, symbol timing | `±fs / (2 · spacing)` |
| `BiasTone(...)` | position of a transmitted CW tone | a tone in the spectrum | anywhere in the band |

1. blind estimation and its lock range;
2. accuracy of the two blind estimators;
3. coarse blind plus fine pilot-aided estimation;
4. a pilot tone;
5. a drifting offset, tracked blockwise.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import commkit as ck
from commkit import RRC, Constellation
from commkit.backend import to_device
from commkit.frequency import BiasTone, MengaliMorelli, MthPower, PilotSymbols

DEVICE = "gpu" if ck.backend.is_cupy_available() else "cpu"
print(DEVICE)

ck.plotting.apply_default_theme()
C16 = Constellation.qam(16)


def host(a):
    return to_device(a, "cpu")


tx = ck.generate(
    C16,
    num_symbols=2**16,
    symbol_rate=32e9,
    sps=2,
    pulse=RRC(rolloff=0.1),
    rng=1,
    device=DEVICE,
)


def link(df, *, esn0_db=15.0, linewidth=100e3, rng=2):
    """Frequency offset, laser phase noise and AWGN on the 2-sps waveform."""
    rx = ck.spectral.shift_frequency(tx, frequency=df)
    if linewidth:
        rx = ck.impairments.apply_phase_noise(rx, linewidth=linewidth, rng=rng)
    return ck.impairments.apply_awgn(rx, esn0_db=esn0_db, rng=rng + 1)


def applied(df):
    """The offset shift_frequency actually applies (snapped to the FFT grid)."""
    return ck.spectral.grid_frequency(
        df, sampling_rate=tx.sampling_rate, num_samples=tx.samples.shape[-1]
    )

## 1. Blind estimation: the M-th power

For a constellation with rotational symmetry `M` (4 for QAM), `x^M` turns
every symbol into nearly the same phase, so `x^M` contains a spectral line
at `M·Δf`.  `MthPower` finds the FFT peak and refines it between bins.  The
default `M` comes from the `Signal`'s constellation.  The estimate is a
`FrequencyOffsetEstimate` with the value and diagnostics for the plots.

FOE runs on the received waveform *before* the matched filter: with a large
offset the signal sits off-centre, and the matched filter would cut part of
it away.

In [ ]:
DF = 1.2345e9
rx = link(DF)
est = ck.frequency.estimate_frequency_offset(rx, MthPower())
print(f"true {applied(DF) / 1e6:.3f} MHz, estimated {float(est.value) / 1e6:.3f} MHz")
ck.plotting.plot_frequency_offset_spectrum(est, show=True)

corrected = ck.filtering.matched_filter(ck.frequency.correct_frequency_offset(rx, est))
fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
ck.plotting.plot_constellation(
    ck.multirate.decimate_to_symbol_rate(ck.filtering.matched_filter(rx)),
    ax=axes[0],
    title="1.23 GHz offset",
)
ck.plotting.plot_constellation(
    ck.multirate.decimate_to_symbol_rate(corrected).samples[..., :2000],
    ax=axes[1],
    title="Corrected (first 2000 symbols)",
    constellation=C16,
)
plt.show()

The corrected constellation is still slightly rotated: what remains is the
laser phase noise and a small residual offset, which carrier-phase recovery
([13](13-blind-carrier-phase-recovery.ipynb)) removes.

**Lock range.**  The line sits at `M·Δf`, which the FFT can only tell
apart modulo the sampling rate: offsets beyond `±fs/(2M)` (±8 GHz here, at
64 GS/s) alias.  A 9 GHz offset reads as `9 - 64/4 = -7` GHz.

In [ ]:
for df in (6e9, 9e9):
    e = ck.frequency.estimate_frequency_offset(link(df), MthPower())
    print(
        f"true {df / 1e9:4.1f} GHz -> estimated {float(e.value) / 1e9:+.3f} GHz "
        f"(lock range ±{tx.sampling_rate / 8 / 1e9:.0f} GHz)"
    )

## 2. Accuracy

`MengaliMorelli` combines the phase of the autocorrelation of `x^M` over
many lags with optimal weights and reaches the Cramér-Rao bound; the
M-th power with interpolation is close.  Both improve with the record
length and the SNR.  The spread over independent noise realizations,
first without phase noise:

In [ ]:
for esn0 in (10, 20):
    errors = {"MthPower": [], "MengaliMorelli": []}
    for trial in range(8):
        r = link(DF, esn0_db=esn0, linewidth=0, rng=10 + 2 * trial)
        for name, method in (
            ("MthPower", MthPower()),
            ("MengaliMorelli", MengaliMorelli()),
        ):
            errors[name].append(
                float(ck.frequency.estimate_frequency_offset(r, method).value)
                - applied(DF)
            )
    print(
        f"Es/N0 {esn0} dB: "
        + ", ".join(
            f"{k} RMS error {np.sqrt(np.mean(np.square(v))) / 1e3:6.2f} kHz"
            for k, v in errors.items()
        )
    )

With laser phase noise the question itself gets fuzzy: a Wiener phase over
a record of duration `T` has a random mean frequency of standard deviation
about `sqrt(Δν / (2π T))`, 90 kHz for 100 kHz over 2 µs.  Below that level
"the offset" is part of the phase noise, and carrier recovery tracks it.

In [ ]:
errs = []
for trial in range(8):
    r = link(DF, esn0_db=20, rng=40 + 2 * trial)
    errs.append(
        float(ck.frequency.estimate_frequency_offset(r, MengaliMorelli()).value)
        - applied(DF)
    )
T = tx.samples.shape[-1] / tx.sampling_rate
print(
    f"100 kHz laser: RMS error {np.sqrt(np.mean(np.square(errs))) / 1e3:.0f} kHz, "
    f"sqrt(Δν / 2πT) = {np.sqrt(100e3 / (2 * np.pi * T)) / 1e3:.0f} kHz"
)

## 3. Coarse blind, fine with pilots

Known pilot symbols give the phase directly: `angle(r · conj(s))` grows by
`2π Δf T_p` between pilots `T_p` apart, and a line fit gives `Δf`.  This
needs symbol timing (the pilots must be at known indices) and it aliases
beyond `±1/(2 T_p)`: with a pilot every 32 symbols at 32 GBd, ±500 MHz.  The
usual receiver therefore removes the bulk blindly and refines with pilots.
Here every 32nd symbol of the record serves as a pilot, on a link without
phase noise so that the offset is well defined.

In [ ]:
rx0 = link(DF, linewidth=0)
blind = ck.frequency.estimate_frequency_offset(rx0, MthPower())
y = ck.multirate.decimate_to_symbol_rate(ck.filtering.matched_filter(rx0))
pilots = np.arange(0, y.samples.shape[-1], 32)
pilot_method = PilotSymbols(indices=pilots, values=host(y.reference.symbols)[pilots])

direct = ck.frequency.estimate_frequency_offset(y, pilot_method)
print(
    f"pilots alone on a {applied(DF) / 1e9:.3f} GHz offset: {float(direct.value) / 1e6:9.3f} MHz (aliased)"
)

coarse = ck.multirate.decimate_to_symbol_rate(
    ck.filtering.matched_filter(ck.frequency.correct_frequency_offset(rx0, blind))
)
fine = ck.frequency.estimate_frequency_offset(coarse, pilot_method)
total = float(blind.value) + float(fine.value)
print(
    f"blind {float(blind.value) / 1e6:.4f} MHz + pilot residual {float(fine.value) / 1e3:+.2f} kHz "
    f"= {total / 1e6:.4f} MHz (true {applied(DF) / 1e6:.4f})"
)
ck.plotting.plot_pilot_phase_estimate(fine, sampling_rate=y.sampling_rate, show=True)

## 4. A pilot tone

If the transmitter adds a CW tone at a known frequency `f_t`
([02](02-spectral-analysis.ipynb) · section 5), the receiver finds it at
`f_t + Δf`, independent of the modulation and over the whole band.
`BiasTone` locates the line inside a search window; the offset is the
distance from where the tone was sent.

In [ ]:
F_TONE = 20e9
with_tone = ck.spectral.add_pilot_tone(tx, frequency=F_TONE, power_ratio_db=-15)
r = ck.impairments.apply_awgn(
    ck.spectral.shift_frequency(with_tone, frequency=DF), esn0_db=15, rng=5
)
tone = ck.frequency.estimate_frequency_offset(
    r, BiasTone(target_frequency=F_TONE, search_band=4e9)
)
sent = ck.spectral.grid_frequency(
    F_TONE, sampling_rate=tx.sampling_rate, num_samples=tx.samples.shape[-1]
)
print(
    f"tone found at {float(tone.value) / 1e9:.4f} GHz -> offset {(float(tone.value) - sent) / 1e6:.3f} MHz "
    f"(true {applied(DF) / 1e6:.3f})"
)

## 5. A drifting offset

Lasers drift: temperature and current changes move the offset during a
capture.  With `block_size`, every estimator measures per block, and
`correct_frequency_offset` interpolates the block values and integrates
them into a phase trajectory.  Here the offset sweeps from 300 MHz to
700 MHz over the 2 µs record.

In [ ]:
n = tx.samples.shape[-1]
t = np.arange(n) / tx.sampling_rate
sweep = 300e6 + 400e6 * t / t[-1]
phase = 2 * np.pi * np.cumsum(sweep) / tx.sampling_rate
drifting = tx.replace(
    samples=tx.samples
    * ck.backend.to_device(np.exp(1j * phase).astype(np.complex64), DEVICE)
)
drifting = ck.impairments.apply_awgn(drifting, esn0_db=15, rng=6)

blockwise = ck.frequency.estimate_frequency_offset(drifting, MthPower(block_size=8192))
axes = ck.plotting.plot_frequency_offset_blockwise_result(
    blockwise, num_samples=n, sampling_rate=tx.sampling_rate
)[1]
axes[0].plot(np.arange(n), sweep / 1e3, "--", label="true offset")
axes[0].legend()
plt.show()

for name, how in (("one estimate", MthPower()), ("blockwise", blockwise)):
    z = ck.multirate.decimate_to_symbol_rate(
        ck.filtering.matched_filter(
            ck.frequency.correct_frequency_offset(drifting, how)
        )
    )
    resid = np.unwrap(np.angle(host(z.samples) * np.conj(host(z.reference.symbols))))
    print(f"{name:13s}: residual phase wander over the record {np.ptp(resid):6.1f} rad")

**Summary**

* Blind FOE (`MthPower`, `MengaliMorelli`) needs no data, locks within
  `±fs/(2M)`, and leaves a residual for carrier recovery.
* Pilots give a fine estimate with a narrow range: use them after a coarse
  blind step.
* A transmitted tone is found anywhere in the band, independent of the
  modulation.
* `block_size=` tracks a drifting offset; `correct_frequency_offset`
  integrates it into a smooth phase.

**Next:** [13 · Blind carrier-phase recovery](13-blind-carrier-phase-recovery.ipynb).